# Fast existence check for an odd cycle with torsion

`odd_cycle_check.py` answers *"is there a vector of odd augmentation in the 2-saturation of $\mathrm{colspan}_{\mathbb Z}(M)$?"*
without constructing one, in a single integer kernel plus one rank over $\mathbb F_2$:

1. compute a full integer basis of $\ker_{\mathbb Z}(M^{T})$ (the left kernel of $M$);
2. write the basis vectors as the rows of $A$;
3. reduce $A$ modulo $2$;
4. check whether the rank grows when the all-ones row is appended:
$$\operatorname{rank}_{\mathbb F_2}\begin{pmatrix}A\bmod 2\\ 1\ \cdots\ 1\end{pmatrix} \;\stackrel{?}{=}\; \operatorname{rank}_{\mathbb F_2}(A\bmod 2)+1.$$

If the rank grows the vector exists; if not, every vector of the 2-saturation has even augmentation.

In [1]:
import sys, os, time
sys.path.append(os.getcwd())

from odd_cycle_check import check, has_odd_torsion_on, has_odd_torsion_vector
from odd_torsion import sphere_vectors_z3, build_relation_matrix, find_odd_torsion_vector

## The family $T$

Even squarefree $d$ not represented by the triangle form $2(a^2+ab+b^2)$, i.e. outside Noble's triangle theorem.

In [2]:
S = sorted([2*(a**2 + a*b + b**2) for a in range(-100, 100) for b in range(-100, 100)])
S = [ZZ(x).squarefree_part() for x in S]
#T = [ZZ(x).squarefree_part() for x in range(2, 1000, 2)]
T = [ZZ(x).squarefree_part() for x in range(2, 1000, 2)]
T = sorted([x for x in set(T) - set(S) if x % 2 == 0])
#T = sorted([x for x in set(T) if x % 2 == 0])
print(len(T), 'values:', *T, sep=' ')

127 values: 10 22 30 34 46 58 66 70 82 94 102 106 110 118 130 138 142 154 166 170 174 178 190 202 210 214 226 230 238 246 262 274 282 286 290 298 310 318 322 330 334 346 354 358 370 374 382 390 394 406 410 418 426 430 442 454 462 466 470 478 498 502 506 510 514 526 530 534 538 562 570 574 586 590 598 606 610 622 634 638 642 646 658 670 678 682 690 694 706 710 714 718 730 742 754 766 770 778 782 786 790 802 814 822 826 830 838 858 862 870 874 886 890 894 898 902 910 922 930 934 946 958 966 970 982 986 994


## Screening all of $T$ at $D=9$, in parallel

Each $d$ is an independent check, so the sweep forks one task per $d$ (sage `@parallel`),
each reporting how many orbits that $d$ needs. A `tqdm` bar tracks how many distances are
done and how many are left (with an ETA); the per-$d$ lines go through `bar.write` so they
do not scramble the bar.

The worker catches its own exceptions on purpose: an unhandled exception inside a forked task
propagates and aborts the **whole** loop, so one bad $d$ would otherwise kill the entire sweep.
Results come back out of order, so they are collected in a dict and summarised at the end.

In [3]:
import multiprocessing, traceback
from sage.all import parallel
from tqdm.auto import tqdm

D = 9
NCPUS = min(len(T), multiprocessing.cpu_count())
print(f"|T|={len(T)}, D={D}, NCPUS={NCPUS}")

@parallel(ncpus=NCPUS)
def check_one(d, D):
    from odd_cycle_check import check
    try:
        r = check(d, D, verbose=False, full_sphere=False)
        return {"exists": r["exists"], "size_S": r["size_S"],
                "orbits_needed": r["orbits_needed"],
                "points_needed": r["points_needed"],
                "seconds": r["seconds"], "error": None}
    except Exception:
        return {"exists": False, "size_S": None, "orbits_needed": None,
                "points_needed": None, "seconds": 0.0,
                "error": traceback.format_exc()}

t_start = time.time()
screen = {}
# int(...) everywhere tqdm sees a number: the Sage preparser turns literals
# into Integer, and Integer/Integer is a Rational, which tqdm cannot format
bar = tqdm(total=int(len(T)), desc=f"D={D}", unit="d")

for (args, kwargs), out in check_one([(d, D) for d in T]):
    d = int(args[0])
    if isinstance(out, str):   # worker died before its own except could run
        out = {"exists": False, "size_S": None, "orbits_needed": None,
               "points_needed": None, "seconds": 0.0, "error": out}
    screen[d] = out

    if out["error"]:
        bar.write(f"d={d:>4}  ERROR\n{out['error']}")
    elif out["exists"]:
        bar.write(f"d={d:>4}  |S|={out['size_S']:>5}  exists: "
                  f"{out['orbits_needed']} orbit(s), {out['points_needed']} pts  "
                  f"{out['seconds']:.2f}s")
    else:
        bar.write(f"d={d:>4}  |S|={out['size_S']:>5}  NOT FOUND  "
                  f"{out['seconds']:.2f}s")

    n_ok = int(sum(1 for r in screen.values() if r["exists"]))
    n_err = int(sum(1 for r in screen.values() if r["error"]))
    bar.set_postfix(found=n_ok, missing=int(len(screen)) - n_ok - n_err,
                    errors=n_err)
    bar.update()

bar.close()

ok   = sorted(d for d, r in screen.items() if r["exists"])
bad  = sorted(d for d, r in screen.items() if not r["exists"] and not r["error"])
errs = sorted(d for d, r in screen.items() if r["error"])
print()
print(f"wall time {time.time()-t_start:.1f}s on {NCPUS} core(s)")
print(f"exists at D={D} ({len(ok)}/{len(T)}): {ok}")
if bad:
    print(f"NOT at D={D}: {bad}")
if errs:
    print(f"errors: {errs}")

|T|=127, D=9, NCPUS=16


D=9:   0%|          | 0/127 [00:00<?, ?d/s]

d=  94  |S|= 1632  exists: 4 orbit(s), 192 pts  1.70s
d=  82  |S|=  816  exists: 5 orbit(s), 216 pts  2.29s
d=  10  |S|=  408  exists: 5 orbit(s), 216 pts  2.56s
d=  34  |S|=  816  exists: 6 orbit(s), 240 pts  4.70s
d=  46  |S|=  816  exists: 6 orbit(s), 288 pts  7.79s
d=  70  |S|=  816  exists: 6 orbit(s), 288 pts  8.50s
d=  22  |S|=  408  exists: 7 orbit(s), 264 pts  9.34s
d=  30  |S|=  624  exists: 7 orbit(s), 336 pts  17.22s
d= 170  |S|= 1296  exists: 8 orbit(s), 336 pts  16.89s
d=  58  |S|=  408  exists: 8 orbit(s), 360 pts  25.78s
d=  66  |S|= 1248  exists: 8 orbit(s), 384 pts  31.61s
d= 130  |S|=  816  exists: 9 orbit(s), 384 pts  32.29s
d= 226  |S|= 1632  exists: 5 orbit(s), 216 pts  3.25s
d= 202  |S|= 1224  exists: 8 orbit(s), 360 pts  21.52s
d= 174  |S|= 1872  exists: 8 orbit(s), 384 pts  31.65s
d= 106  |S|= 1224  exists: 9 orbit(s), 408 pts  47.98s
d= 110  |S|= 1296  exists: 9 orbit(s), 432 pts  51.62s
d= 118  |S|= 1224  exists: 10 orbit(s), 456 pts  56.61s
d= 154  |S|= 1632